# 02 — Data Preprocessing Pipeline
### Climate-Smart Agriculture: Bi-Seasonal Paddy Yield Forecasting
**Module:** IT41033 Nature Inspired Algorithms (NIA)

---
This notebook demonstrates the end-to-end preprocessing pipeline:
1. **Cleaning**: Deduplication, outlier treatment via IQR winsorization, missing value management.
2. **Transformation**: Seasonal categorical binarization ($S_{binary} \in \{0, 1\}$).
3. **Scaling**: Z-score standardization ($StandardScaler$) fitted exclusively on training data to prevent temporal lookahead leakage.
4. **Export**: Exporting `data/processed/model_ready.csv`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import run_preprocessing_pipeline, winsorize_series_iqr

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 150})

## 1. Execute Preprocessing Pipeline

In [ ]:
prep_results = run_preprocessing_pipeline(train_ratio=0.85, save_outputs=True)
df_model = prep_results['df']
print(f"Processed Data Shape: {df_model.shape}")
display(df_model.head(10))

## 2. Before vs After: Outlier Winsorization Comparison

In [ ]:
raw_df = pd.read_excel('../data/raw/kaggle_rice_climate/rice new one.xlsx')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(y=raw_df['Temperature(°C)'], ax=axes[0], color='#e74c3c')
axes[0].set_title('Raw Temperature (°C) - Showing Outliers')

sns.boxplot(y=df_model['temperature_c'], ax=axes[1], color='#2ecc71')
axes[1].set_title('Winsorized Temperature (°C) - Capped via IQR')
plt.tight_layout()
plt.show()

## 3. Standardization Distribution Verification

In [ ]:
scaled_cols = prep_results['scaled_features']
train_mask = prep_results['train_mask']

print("Training Set Scaled Feature Means (should be ~0):")
print(df_model.loc[train_mask, scaled_cols].mean().round(4))

print("\nTraining Set Scaled Feature Std Dev (should be ~1):")
print(df_model.loc[train_mask, scaled_cols].std().round(4))

df_model.loc[train_mask, scaled_cols].hist(figsize=(14, 8), bins=20, edgecolor='white')
plt.suptitle('Standardized Feature Distributions (Training Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Preprocessing Summary Table

In [ ]:
summary_data = {
    'Attribute': ['year', 'season', 'rainfall_mm', 'temperature_c', 'gdp_billion_usd', 'inflation_pct', 'production_000_mt'],
    'Type': ['Temporal Identifier', 'Categorical', 'Continuous Metric', 'Continuous Metric', 'Continuous Metric', 'Continuous Metric', 'Target Variable'],
    'Cleaning Applied': ['Deduplicated', 'Standardized naming', 'IQR Winsorization', 'IQR Winsorization', 'IQR Winsorization', 'IQR Winsorization', 'Validated against DCS'],
    'Transformation': ['Integer key', 'Binary Encoding (0/1)', 'Z-Score Standardization', 'Z-Score Standardization', 'Z-Score Standardization', 'Z-Score Standardization', 'Target (000 Mt)']
}
summary_table = pd.DataFrame(summary_data)
display(summary_table)